In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set up paths
SAMPLES_DIR = Path("samples")
MASTER_FILE = SAMPLES_DIR / "Expense Report Relating to deals 01-01-26 to 6-30-26_V8 - COMBINED.xlsx"
BSNY_CONCUR_FILE = SAMPLES_DIR / "BSNY - SAP & Concur Repoort May 2025 - June 2026.xlsx"
SANCAP_CONCUR_FILE = SAMPLES_DIR / "SanCap - Expense Report USA May 2025 - JUNE 2026.xlsx"


# ------------------------------------------------------------------------------------------
print("✓ Dependencies loaded")
print(f"✓ Sample files directory: {SAMPLES_DIR}")
print(f"✓ Files ready to process")

In [ ]:
# STEP 10A: COM refresh preflight (this cell does not modify any workbook)
from pathlib import Path
import shutil
import win32com.client as win32

OUTPUT_DIR = Path("outputs")
OUTPUT_MASTER_FILE = OUTPUT_DIR / f"{MASTER_FILE.stem}_paste_concur{MASTER_FILE.suffix}"

required_files = [MASTER_FILE, BSNY_CONCUR_FILE, SANCAP_CONCUR_FILE]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError("Required workbook(s) not found:\n" + "\n".join(missing_files))

OUTPUT_DIR.mkdir(exist_ok=True)
print("STEP 10A: COM Refresh Preflight")
print(f"Master input: {MASTER_FILE.name}")
print(f"Output file:  {OUTPUT_MASTER_FILE}")
print("The existing output file will be overwritten after Step 10B validation succeeds.")

In [ ]:
# STEP 10B: Create the refreshed master workbook using Excel COM
# Run Step 10A first. This code only writes to OUTPUT_MASTER_FILE, never MASTER_FILE.

MASTER_SHEET_NAME = "Concur Report"
BSNY_SHEET_NAME = "Concur"
HELPER_HEADERS = [
    "First & Last Name",
    "CC Expense is Mapped to",
    "LOB",
    "In Scope",
    "Error",
]
SOURCE_TO_MASTER = {
    "Custom 41 - Name": "Client Name",
    "Custom 42 - Name": "Project Name",
    "Custom 43 - Name": "Epense Name",
}
YEAR_HEADER = "Year"
EXPENSE_HEADER = "Expense Amount (reimbursement currency)"
XL_UP = -4162
XL_TO_LEFT = -4159
XL_CALCULATION_MANUAL = -4135
XL_CALCULATION_AUTOMATIC = -4105
XL_PASTE_VALUES = -4163
XL_SHEET_VERY_HIDDEN = 2


def normalise_header(value):
    """Return a consistent Excel header value, rejecting blank headers."""
    if value is None:
        return ""
    return str(value).strip()


def read_headers(worksheet):
    """Read row 1 and return every column position for each non-blank header."""
    last_column = worksheet.Cells(1, worksheet.Columns.Count).End(XL_TO_LEFT).Column
    raw_headers = worksheet.Range(worksheet.Cells(1, 1), worksheet.Cells(1, last_column)).Value2
    header_values = list(raw_headers[0]) if isinstance(raw_headers, tuple) else [raw_headers]
    headers = {}

    for column_number, raw_header in enumerate(header_values, start=1):
        header = normalise_header(raw_header)
        if not header:
            continue
        headers.setdefault(header, []).append(column_number)
    return headers


def unique_header_column(headers, header, worksheet):
    """Return one required column, rejecting missing or repeated structural headers."""
    columns = headers.get(header, [])
    if len(columns) != 1:
        description = "missing" if not columns else f"repeated in columns {columns}"
        raise ValueError(f"Required header '{header}' is {description} in '{worksheet.Name}'.")
    return columns[0]


def last_used_row(worksheet):
    """Find the last row containing a value or formula."""
    return worksheet.Cells(worksheet.Rows.Count, 1).End(XL_UP).Row


def column_values(worksheet, column_number, last_row):
    """Return values from Excel rows 2:last_row as a Python list."""
    if last_row < 2:
        return []
    raw_values = worksheet.Range(worksheet.Cells(2, column_number), worksheet.Cells(last_row, column_number)).Value2
    if last_row == 2:
        return [raw_values]
    return [row[0] for row in raw_values]


def is_reporting_year(value, year=2026):
    """Support Excel numeric and text representations of the reporting year."""
    try:
        return float(value) == float(year)
    except (TypeError, ValueError):
        return str(value).strip() == str(year)


def source_rows_for_year(worksheet, headers):
    """Return Excel row numbers that belong to the 2026 reporting period."""
    source_last_row = last_used_row(worksheet)
    year_column = unique_header_column(headers, YEAR_HEADER, worksheet)
    years = column_values(worksheet, year_column, source_last_row)
    return [row_number for row_number, value in enumerate(years, start=2) if is_reporting_year(value)]


def values_for_rows(worksheet, column_number, rows):
    """Read one source column and keep only the requested Excel row numbers."""
    if not rows:
        return []
    values = column_values(worksheet, column_number, max(rows))
    return [values[row_number - 2] for row_number in rows]


def paste_column_from_staging(staging_sheet, worksheet, start_row, column_number, values):
    """Stage values on a blank sheet, then paste them into one master column."""
    if not values:
        return
    staging_range = staging_sheet.Range(
        staging_sheet.Cells(1, 1),
        staging_sheet.Cells(len(values), 1),
    )
    staging_range.Value2 = tuple((value,) for value in values)
    staging_range.Copy()
    worksheet.Range(
        worksheet.Cells(start_row, column_number),
        worksheet.Cells(start_row + len(values) - 1, column_number),
    ).PasteSpecial(Paste=XL_PASTE_VALUES)


def numeric_total(values):
    """Sum values while treating blank Excel cells as zero."""
    total = 0.0
    for value in values:
        if value not in (None, ""):
            total += float(value)
    return total


def build_column_mappings(master_headers, bsny_headers, sancap_headers):
    """Map each non-helper master column to the corresponding source-header occurrence."""
    master_to_source = {target: source for source, target in SOURCE_TO_MASTER.items()}
    source_occurrences = {}
    mappings = []

    for master_header, master_columns in master_headers.items():
        if master_header in HELPER_HEADERS:
            continue
        source_header = master_to_source.get(master_header, master_header)
        for master_column in master_columns:
            occurrence = source_occurrences.get(source_header, 0)
            source_occurrences[source_header] = occurrence + 1
            for source_name, source_headers in (("BSNY", bsny_headers), ("SanCap", sancap_headers)):
                available_columns = source_headers.get(source_header, [])
                if len(available_columns) <= occurrence:
                    raise ValueError(
                        f"{source_name} is missing occurrence {occurrence + 1} of source header "
                        f"'{source_header}' required for master column {master_column}."
                    )
            mappings.append((
                master_column,
                bsny_headers[source_header][occurrence],
                sancap_headers[source_header][occurrence],
            ))

    return mappings


def next_available_output_path(output_file):
    """Avoid overwriting a prior refreshed workbook, including one open in Excel."""
    candidate = output_file
    suffix_number = 1
    while candidate.exists():
        candidate = output_file.with_name(
            f"{output_file.stem}_{suffix_number}{output_file.suffix}"
        )
        suffix_number += 1
    return candidate


def resize_containing_table(worksheet, column_number, final_row):
    """Resize the table containing a target column before bulk data is written."""
    for table_index in range(1, worksheet.ListObjects.Count + 1):
        table = worksheet.ListObjects(table_index)
        first_column = table.Range.Column
        last_column = first_column + table.Range.Columns.Count - 1
        if first_column <= column_number <= last_column:
            table.Resize(worksheet.Range(
                worksheet.Cells(table.Range.Row, first_column),
                worksheet.Cells(final_row, last_column),
            ))
            return table.Name
    return None


excel = None
master_workbook = None
bsny_workbook = None
sancap_workbook = None
staging_sheet = None
refresh_succeeded = False
auto_fill_formulas_in_lists = None

try:
    if not OUTPUT_MASTER_FILE.parent.exists():
        raise RuntimeError("Run Step 10A before Step 10B.")

    # Create a new output copy before Excel opens any workbook.
    base_output_file = OUTPUT_DIR / f"{MASTER_FILE.stem}_REFRESHED{MASTER_FILE.suffix}"
    OUTPUT_MASTER_FILE = next_available_output_path(base_output_file)
    shutil.copy2(MASTER_FILE, OUTPUT_MASTER_FILE)

    excel = win32.DispatchEx("Excel.Application")
    excel.Visible = False
    excel.DisplayAlerts = False
    excel.ScreenUpdating = False
    excel.EnableEvents = False

    master_workbook = excel.Workbooks.Open(str(OUTPUT_MASTER_FILE.resolve()))
    bsny_workbook = excel.Workbooks.Open(str(BSNY_CONCUR_FILE.resolve()), ReadOnly=True)
    sancap_workbook = excel.Workbooks.Open(str(SANCAP_CONCUR_FILE.resolve()), ReadOnly=True)
    excel.Calculation = XL_CALCULATION_MANUAL
    auto_fill_formulas_in_lists = excel.AutoCorrect.AutoFillFormulasInLists
    excel.AutoCorrect.AutoFillFormulasInLists = False

    master_sheet = master_workbook.Worksheets(MASTER_SHEET_NAME)
    bsny_sheet = bsny_workbook.Worksheets(BSNY_SHEET_NAME)
    sancap_sheet = sancap_workbook.Worksheets(1)
    staging_sheet = master_workbook.Worksheets.Add(After=master_workbook.Worksheets(master_workbook.Worksheets.Count))
    staging_sheet.Visible = XL_SHEET_VERY_HIDDEN

    master_headers = read_headers(master_sheet)
    bsny_headers = read_headers(bsny_sheet)
    sancap_headers = read_headers(sancap_sheet)

    # Helper, mapped, year, and expense columns are workbook structure and must be unique.
    helper_columns = {
        header: unique_header_column(master_headers, header, master_sheet)
        for header in HELPER_HEADERS
    }
    for target_header in SOURCE_TO_MASTER.values():
        unique_header_column(master_headers, target_header, master_sheet)
    master_year_column = unique_header_column(master_headers, YEAR_HEADER, master_sheet)
    master_expense_column = unique_header_column(master_headers, EXPENSE_HEADER, master_sheet)

    # Direct duplicate headers, such as Employee ID, are matched by first/second/etc. occurrence.
    column_mappings = build_column_mappings(master_headers, bsny_headers, sancap_headers)
    mapped_master_columns = [master_column for master_column, _, _ in column_mappings]
    if len(mapped_master_columns) != len(set(mapped_master_columns)):
        raise AssertionError("More than one source mapping targets the same master column.")

    bsny_rows = source_rows_for_year(bsny_sheet, bsny_headers)
    sancap_rows = source_rows_for_year(sancap_sheet, sancap_headers)
    if not bsny_rows or not sancap_rows:
        raise ValueError("No 2026 data found in one or both source workbooks. No data was cleared.")

    first_data_row = 2
    final_data_row = len(bsny_rows) + len(sancap_rows) + 1
    current_last_row = last_used_row(master_sheet)
    helper_formulas = {
        header: master_sheet.Cells(first_data_row, helper_columns[header]).FormulaR1C1
        for header in HELPER_HEADERS
    }
    missing_formulas = [header for header, formula in helper_formulas.items() if not str(formula).startswith("=")]
    if missing_formulas:
        raise ValueError("The formula template is missing in row 2 for: " + ", ".join(missing_formulas))

    resized_table = resize_containing_table(master_sheet, master_expense_column, final_data_row)
    if resized_table is not None:
        print(f"Resized Excel table: {resized_table}")

    print("STEP 10B: Refreshing Concur Report through Excel COM")
    print(f"BSNY 2026 rows:   {len(bsny_rows):,}")
    print(f"SanCap 2026 rows: {len(sancap_rows):,}")

    # Clear only source-data columns. Helper columns and their formulas are preserved.
    for master_column, _, _ in column_mappings:
        column_number = master_column
        master_sheet.Range(
            master_sheet.Cells(first_data_row, column_number),
            master_sheet.Cells(current_last_row, column_number),
        ).ClearContents()

    # Copy one target column at a time: first BSNY values, then SanCap values.
    expected_expense_values = None
    for master_column, bsny_column, sancap_column in column_mappings:
        bsny_values = values_for_rows(bsny_sheet, bsny_column, bsny_rows)
        sancap_values = values_for_rows(sancap_sheet, sancap_column, sancap_rows)
        combined_values = bsny_values + sancap_values
        paste_column_from_staging(staging_sheet, master_sheet, first_data_row, master_column, combined_values)
        if master_column == master_expense_column:
            expected_expense_values = combined_values
        if expected_expense_values is not None:
            current_expense_values = column_values(master_sheet, master_expense_column, final_data_row)
            if round(numeric_total(current_expense_values), 2) != round(numeric_total(expected_expense_values), 2):
                differences = [
                    (index + first_data_row, expected, actual)
                    for index, (expected, actual) in enumerate(zip(expected_expense_values, current_expense_values))
                    if expected != actual
                ]
                raise AssertionError(
                    f"Writing master column {master_column} "
                    f"('{master_sheet.Cells(1, master_column).Value2}') changed the expense amount values."
                    f" Excel formula in its first data cell: "
                    f"{master_sheet.Cells(first_data_row, master_column).Formula!r}"
                    f" Expected first values: {expected_expense_values[:5]!r}; "
                    f"written first values: {current_expense_values[:5]!r}."
                    f" First differing rows: {differences[:5]!r}"
                )

    written_expense_values = column_values(master_sheet, master_expense_column, final_data_row)
    if round(numeric_total(written_expense_values), 2) != round(numeric_total(expected_expense_values), 2):
        raise AssertionError("The expense column changed during source-data writes, before formula calculation.")

    # Remove the source Text format before filling relative helper formulas.
    for helper_header, formula in helper_formulas.items():
        helper_column = helper_columns[helper_header]
        formula_range = master_sheet.Range(
            master_sheet.Cells(first_data_row, helper_column),
            master_sheet.Cells(final_data_row, helper_column),
        )
        formula_range.Clear()
        formula_range.FormulaR1C1 = formula

    excel.Calculation = XL_CALCULATION_AUTOMATIC
    excel.CalculateFullRebuild()

    # Validate the saved-in-memory result before closing Excel.
    expected_rows = len(bsny_rows) + len(sancap_rows)
    actual_rows = final_data_row - first_data_row + 1
    if actual_rows != expected_rows:
        raise AssertionError(f"Expected {expected_rows:,} data rows but wrote {actual_rows:,}.")

    for helper_header in HELPER_HEADERS:
        helper_cell = master_sheet.Cells(final_data_row, helper_columns[helper_header])
        if not helper_cell.HasFormula:
            raise AssertionError(f"Helper formula was not filled to the last row: {helper_header}")

    year_column = master_year_column
    if not is_reporting_year(master_sheet.Cells(first_data_row, year_column).Value2):
        raise AssertionError("The first output row is not a 2026 BSNY record.")
    sancap_start_row = first_data_row + len(bsny_rows)
    if not is_reporting_year(master_sheet.Cells(sancap_start_row, year_column).Value2):
        raise AssertionError("The first SanCap output row is not a 2026 record.")

    expected_bsny_total = numeric_total(
        values_for_rows(bsny_sheet, unique_header_column(bsny_headers, EXPENSE_HEADER, bsny_sheet), bsny_rows)
    )
    expected_sancap_total = numeric_total(
        values_for_rows(sancap_sheet, unique_header_column(sancap_headers, EXPENSE_HEADER, sancap_sheet), sancap_rows)
    )
    expected_expense_total = expected_bsny_total + expected_sancap_total
    output_expense_values = column_values(master_sheet, master_expense_column, final_data_row)
    actual_bsny_total = numeric_total(output_expense_values[:len(bsny_rows)])
    actual_sancap_total = numeric_total(output_expense_values[len(bsny_rows):])
    actual_expense_total = actual_bsny_total + actual_sancap_total
    if round(actual_expense_total, 2) != round(expected_expense_total, 2):
        raise AssertionError(
            "Expense total mismatch. "
            f"BSNY expected/found: {expected_bsny_total:,.2f}/{actual_bsny_total:,.2f}; "
            f"SanCap expected/found: {expected_sancap_total:,.2f}/{actual_sancap_total:,.2f}."
        )

    staging_sheet.Visible = True
    staging_sheet.Delete()
    staging_sheet = None
    master_workbook.Save()
    refresh_succeeded = True
    print(f"Data rows written: {expected_rows:,} (BSNY first, then SanCap)")
    print(f"Expense total: ${actual_expense_total:,.2f}")
    print(f"Saved refreshed file: {OUTPUT_MASTER_FILE}")
    

finally:
    if bsny_workbook is not None:
        bsny_workbook.Close(SaveChanges=False)
    if sancap_workbook is not None:
        sancap_workbook.Close(SaveChanges=False)
    if staging_sheet is not None:
        try:
            staging_sheet.Visible = True
            staging_sheet.Delete()
        except Exception:
            pass
    if master_workbook is not None:
        master_workbook.Close(SaveChanges=refresh_succeeded)
    if excel is not None:
        if auto_fill_formulas_in_lists is not None:
            excel.AutoCorrect.AutoFillFormulasInLists = auto_fill_formulas_in_lists
        excel.Quit()
    excel = None
    master_workbook = None
    bsny_workbook = None
    sancap_workbook = None

In [ ]:
# STEP 10C: Publish the validated refresh under the stable output filename
# Run this only after Step 10B completes successfully.

PUBLISHED_OUTPUT_FILE = OUTPUT_DIR / f"{MASTER_FILE.stem}_paste_concur{MASTER_FILE.suffix}"

if not OUTPUT_MASTER_FILE.exists():
    raise FileNotFoundError(
        "The validated Step 10B output was not found. Run Step 10B before publishing."
    )

shutil.copy2(OUTPUT_MASTER_FILE, PUBLISHED_OUTPUT_FILE)
OUTPUT_MASTER_FILE = PUBLISHED_OUTPUT_FILE

print(f"Published refreshed file: {OUTPUT_MASTER_FILE}")
print("An existing _paste_concur.xlsx file was overwritten.")

In [ ]:
# STEP 10D: Rebuild helper cells as calculated Excel formulas
# Run after Step 10C. This changes only the five helper columns.

excel = None
output_workbook = None
repair_succeeded = False

try:
    excel = win32.DispatchEx("Excel.Application")
    excel.Visible = False
    excel.DisplayAlerts = False
    excel.ScreenUpdating = False

    output_workbook = excel.Workbooks.Open(str(PUBLISHED_OUTPUT_FILE.resolve()))
    output_sheet = output_workbook.Worksheets(MASTER_SHEET_NAME)
    output_headers = read_headers(output_sheet)
    output_last_row = last_used_row(output_sheet)

    for helper_header in HELPER_HEADERS:
        helper_column = unique_header_column(output_headers, helper_header, output_sheet)
        helper_range = output_sheet.Range(
            output_sheet.Cells(2, helper_column),
            output_sheet.Cells(output_last_row, helper_column),
        )
        formula_template = output_sheet.Cells(2, helper_column).FormulaR1C1
        if not str(formula_template).startswith("="):
            raise ValueError(f"Missing formula template in '{helper_header}'.")

        # The source helper cells are Text-formatted, which makes Excel display
        # reassigned formulas literally. Clear removes that format before filling.
        helper_range.Clear()
        helper_range.FormulaR1C1 = formula_template

        first_cell = output_sheet.Cells(2, helper_column)
        if not first_cell.HasFormula:
            raise AssertionError(f"'{helper_header}' was not converted into an Excel formula.")

    excel.CalculateFullRebuild()
    output_workbook.Save()
    repair_succeeded = True
    print(f"Helper formulas recalculated in: {PUBLISHED_OUTPUT_FILE}")
finally:
    if output_workbook is not None:
        output_workbook.Close(SaveChanges=repair_succeeded)
    if excel is not None:
        excel.Quit()